# 01 — External DB setup (CultPass)

This notebook initializes and seeds `data/external/cultpass.db` — the database owned by **CultPass**, UDA-Hub's first customer. It contains `accounts`, `users`, and `bookings`. UDA-Hub's own agents only ever touch this data through tools (`agentic/tools/account_tools.py`, `refund_tool.py`), never directly — this notebook is purely for standing up the demo data.

Run all cells top to bottom. It's safe to re-run: seeding is skipped if the DB already has data.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
print("External DB path:", config.EXTERNAL_DB_PATH)

In [ ]:
from data.external.seed_accounts import seed

seed()

## Verify the data

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(config.EXTERNAL_DB_PATH)

print("Accounts:")
display(pd.read_sql("SELECT id, account_name, plan_tier, status, billing_cycle FROM accounts", conn))

print("Users:")
display(pd.read_sql("SELECT id, account_id, full_name, email, role FROM users", conn))

print("Bookings:")
display(pd.read_sql(
    "SELECT id, user_id, experience_name, amount_usd, booking_date, status FROM bookings ORDER BY booking_date",
    conn,
))

conn.close()

In [ ]:
# Sanity check via the actual tool the agents will use (not raw SQL) —
# proves the abstraction in agentic/tools/account_tools.py works end to end.
from agentic.tools.account_tools import account_lookup_tool

account_lookup_tool.invoke({"email": "lan.nguyen@example.com"})